# IOC Cleanup Notebook

This notebook processes, cleans, and visualizes sea level data from IOC (Intergovernmental Oceanographic Commission) stations. It includes tools for interactive data exploration, flagging anomalies, and isolating storm surges.

## Key Features:
 * **Data Download**: Fetches sea level data from selected IOC stations.
 * **Data Cleaning**: Applies transformations to clean raw data.
 * **Interactive Visualization**: Uses Holoviews and Panel for interactive plotting and flagging.
 * **Flagging System**: Supports flags like `wip`, `skip`, `segments`, and `tsunami` for data quality control.
 * **Surge Calculation**: Detides data to isolate storm surges.
 * **Post-Analysis**: Evaluates data completeness and visualizes results on a map.

## How to Use:
 * **Download Data**: Fetch raw data using searvey.
 * **Clean Data**: Apply transformations from ../transformations.
 * **Visualize & Flag**: Use the interactive dashboard to explore and flag data.
 * **Analyze**: Review flagged data, calculate surges, and explore post-analysis results.

## Requirements:
 * ~15GB of free disk space.
 * Python libraries listed in the imports section.

## 0 - Imports

This section imports the necessary Python libraries for data processing, visualization, and analysis. Key libraries include:
- **Pandas** for data manipulation.
- **Holoviews** and **Panel** for interactive visualizations.
- **Searvey** for fetching IOC station data.
- **UTide** for tidal analysis and surge calculations.

In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import panel as pn
import hvplot.pandas
import holoviews as hv
import pathlib
import utide 
from holoviews.streams import RangeXY,Selection1D
from holoviews.operation.datashader import rasterize

import ioc_cleanup as C

hv.extension('bokeh')
pn.extension()

## 1 - Functions

This section defines core functions for:
- **Reading Data**: Loads raw data from specific stations and sensors.
- **Cleaning Data**: Applies transformations to clean the raw data.
- **Surge Calculation**: Detides the data to isolate storm surges using **UTide**.

In [ ]:
def read(station: str, sensor: str) -> pd.DataFrame:
    df = pd.read_parquet(f"../raw/{station}_{sensor}.parquet")
    return df[sensor]

def clean(station: str, sensor: str) -> pd.DataFrame:
    df = pd.read_parquet(f"../raw/{station}_{sensor}.parquet")
    trans = C.load_transformation_from_path("../transformations/" + station + "_" + sensor + ".json")
    return C.transform(df, trans)[sensor]

def surge(ts: pd.Series, lat: float, rsmp: int = None):
    ts0 = ts.copy()
    OPTS = {
        "constit": "auto",
        "method": "ols", # ols is faster and good for missing data (Ponchaut et al., 2001)
        "order_constit": "frequency",
        "Rayleigh_min": 0.97,
        "lat": lat,
        "verbose": True,
    }
    if rsmp is not None:
        ts = ts.resample(f"{rsmp}min").mean()
        ts = ts.shift(freq=f"{rsmp / 2}min")
    coef = utide.solve(ts.index, ts, **OPTS)
    tidal = utide.reconstruct(ts0.index, coef, verbose = OPTS['verbose'])
    return pd.Series(data=ts0.values - tidal.h, index=ts0.index)

## 2 - Gather IOC data

This section downloads raw sea level data from [**IOC stations**]((https://www.ioc-sealevelmonitoring.org/)
) using the [`searvey`](https://github.com/oceanmodeling/searvey) library. 

### 2.0 - create directories
needed for the rest of the notebook

In [ ]:
!mkdir -p ../data
!mkdir -p ../raw
!mkdir -p ../surge
!mkdir -p html

### 2.1 - View all IOC stations

In [ ]:
ioc_all = C.get_meta()

In [ ]:
all_ioc_ = ioc_all.hvplot.points(
    x= "lon", y='lat', 
    hover_cols = ["ioc_code" ],
    color = "r", 
    line_color = 'k',
    coastline = True,
    tiles = True,
    s = 20).opts(width = 1300, height = 600)
all_ioc_

### 2.2 - View 'IOC Cleanup 2024' subset

A prior selection has been done to find the best IOC candidates for surge model validation.
See the methodology in this notebook: https://tomsail.github.io/static/best_candidates_ioc.html

In [ ]:
ioc_cleanup_2024 = pd.read_csv("https://raw.githubusercontent.com/tomsail/static/refs/heads/master/assets/ioc_cleanup_2024.csv", index_col=0)
all_ = ioc_cleanup_2024.hvplot.points(
    x= "lon", y='lat', 
    color="grey",
    hover_cols = ['ioc_code'], 
    coastline = True, 
    tiles = True,
    line_color='k',
    colorbar=False,
    s = 200)
(all_ * all_ioc_).opts(width = 1300, height = 600)

### 2.3 - download data with `searvey`

This step fetches raw data from IOC stations for the specified time range (2022-01-01 to 2024-12-31). 

The data is saved in the `../data` directory for further processing.

**Note**: 
 * We'll use only 2 stations for this notebook: `PA07` (Palermo) and `SA16` (Salerno). 
 * Use `ioc_cleanup_2024` for the whole dataset

In [ ]:
ioc_cleanup_2024_test = ioc_cleanup_2024[ioc_cleanup_2024.ioc_code.isin(["PA07", "SA16"])]
ioc_cleanup_2024_test

In [ ]:
dfs = C.download_raw(ioc_cleanup_2024_test.ioc_code.values,"2022-01-01", "2024-12-31")
for station, df in dfs.items(): 
    df.to_parquet(f"../data/{station}.parquet")

### 2.4 - extract sensor data for each tide gauge

This step decouples sensor data (e.g., tide gauge readings) from each station and saves it as separate files in the `../raw` directory. 

**Note**: Use the `ioc_cleanup_2024` dataset for processing all stations.

In [ ]:
for i_s, station in ioc_cleanup_2024_test.iterrows():
    df = pd.read_parquet(f"../data/{station.ioc_code}.parquet")
    for sensor in df.columns:
        df[[sensor]].to_parquet(f"../raw/{station.ioc_code}_{sensor}.parquet")

In [ ]:
pattern = r"raw/([\w\d]+)\.parquet"
stations_sensors = [re.search(pattern, path).group(1) for path in glob.glob("../raw/*parquet")]
f"Total decoupled stations: {len(stations_sensors)}"

## 3 - pre-analysis of the data

This section evaluates the quality and completeness of the downloaded data. Key metrics include:
- **Data Availability**:  calculated with the **detide ratio**: Indicates the completeness of the signal between 2022-2024 (closer to 1 is better).

In [ ]:
all_stats = C._statistics.calc_statistics(ioc_cleanup_2024, pathlib.Path("../raw"))

In [ ]:
all_stats.loc[all_stats.detide_ratio > 1, "detide_ratio"] = 1
all_stats

In [ ]:
completeness = all_stats.hvplot.points(
    x= "lon", y='lat', 
    hover_cols = ['index',"detide_ratio" ],
    color = "detide_ratio", 
    coastline = True,
    tiles = True,
    s = 200, line_color = 'grey', cmap="rainbow_r").opts(width = 1300, height = 600)
completeness

## 4 - Dashboarding part

This section provides an interactive dashboard for:
- Visualizing sea level data (tide or surge)
- Selecting and flagging specific signal segments for cleaning or analysis.

### 4.1 - quick raster visualisation (see the whole signal)
This step generates a rasterized plot of the entire signal for a quick overview of the data. Useful for identifying trends, gaps, or anomalies at a glance.

![raster_example](./assets/raster_example.png)

In [ ]:
rasterize(hv.Curve(read("PA07","rad").dropna().loc["2022-01-01":"2025-01-01"])).opts(width=1300,height=700)

### 4.2 - signal cleaning / detiding

This step allows you to:
 * Select a specific **`station`**, **`sensor`**, and **`YEAR`** for analysis.
 * Clean the signal and detide the data to isolate storm surges.

You need to: 
 * Choose a `station`
 * Choose a `sensor`
 * Choose a `YEAR`: from 2022 to 2024
 * choose a `MONTH`: from 1-12 or 0: all year
 * choose to plot the `SURGE` only or the whole (tidal) signal
 * choose a `YEAR` for the display

![example](./assets/dashboard_example.png)

In [ ]:
station = "PA07" 
sensor = "rad"
YEAR = 2024

r_ = read(station, sensor)
r_ = r_.loc[f"{YEAR}-01-01":f"{YEAR}-12-31"]
c_ = clean(station, sensor)
c_ = c_.loc[f"{YEAR}-01-01":f"{YEAR}-12-31"].dropna()
lat = ioc_cleanup_2024[ioc_cleanup_2024.ioc_code == station].lat.values[0]
s_ = surge(c_, lat, rsmp=2)
s_.columns = [sensor]

In [ ]:
MONTH = 7 # 0 selects the whole year
SURGE = 1
df = s_ if SURGE else c_
if MONTH:
    df = df.loc[pd.Timestamp(YEAR,MONTH,1):pd.Timestamp(YEAR,MONTH,1)+pd.Timedelta(days=31)]
C.select_points(df)

## 5 - Post-analysis of the data

This section reviews the cleaned and flagged data, including:
- Stations marked as **`wip`** (Work in Progress).
- Stations to **`skip`** due to data quality issues.
- Stations with **`tsunami`** events
- Stations with **`segments`**

### 5.0 - List all transformed stations.

In [ ]:
pattern = r"transformations/([\w\d]+)\.json"
stations_sensors = [re.search(pattern, path).group(1) for path in glob.glob("../transformations/*json")]
stations = [ss.split("_")[0] for ss in stations_sensors]

In [ ]:
C.plot_geographic_coverage(ioc_cleanup_2024, stations, "stations transformed out of the IOC 2024 subset")
C.plot_geographic_coverage(ioc_all, stations, "stations transformed out of the whole database")

### 5.1 Stations already done

The 2 followings maps are useful for the cleaning : 
1. A map of stations that have been already processed.
2. A visualization of data completeness for the 2022-2024 period.

In [ ]:
all_stats["processed"] = 0
for station_sensor in sorted(stations_sensors): 
    station, sensor = station_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{station_sensor}.json")
    if t_.end == pd.Timestamp("2025-01-01"):
        all_stats.loc[all_stats.ioc_code == station, "processed"] = 1

processed = all_stats[all_stats.processed == True].hvplot.points(
    x= "lon", y='lat', 
    color="green",
    hover_cols = ['index'], 
    coastline = True, 
    tiles = True, 
    cmap = "rainbow",
    line_color='k',
    colorbar=False,
    label = 'processed',
    s = 200)
(all_*processed).opts(width = 1200, height = 500)
completeness.opts(width = 1200, height = 500)

### 5.2 - Stations left in WIP

Stations flagged as **`wip`** (Work in Progress) require further review due to:
- Complex data segments or steps.
- Uncertainties in processing.
- Excessive cleaning requirements (may also be marked as **`skip`**).

In [ ]:
print("> WIP: ")
all_stats["wip"] = 0 # init
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if t_.wip:
        print("  -", station)
        all_stats.loc[all_stats.ioc_code == station, "wip"] = 1

wip = all_stats[all_stats.wip==True].hvplot.points(
    x= "lon", y='lat', 
    coastline = True, 
    tiles = True, 
    hover_cols = ["index"],
    s = 140, c='yellow' )
(all_ * processed * wip).opts(width = 1200, height = 500)

### 5.3 - stations skipped
Stations excluded from analysis due to data quality issues or insufficient data.

In [ ]:
print("> SKIP: ")
all_stats["skip"] = 0 # init
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if t_.skip:
        print("  -", station)
        all_stats.loc[all_stats.ioc_code == station, "skip"] = 1

skip = all_stats[all_stats.skip==True].hvplot.points(
    x= "lon", y='lat', 
    coastline = True, 
    tiles = True, 
    s = 140, c='r' ).opts(width = 1200, height = 500)
(all_*processed*skip).opts(width = 1200, height = 500)

### 5.4 - Stations with notes

Stations with additional notes or comments regarding data quality, processing, or anomalies.

In [ ]:
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if t_.notes != "":
        print(station, "\t", t_.notes)
        all_stats.loc[all_stats.ioc_code == station, "notes"] = t_.notes

(all_*all_stats[~all_stats.notes.isna()].hvplot.points(
    x= "lon", y='lat', 
    hover_cols = ['notes', 'index' ],
    coastline = True, 
    tiles = True, 
    line_color='k', 
    color='orange',
    s = 100)).opts(width = 1200, height = 500)

### 5.5 - Stations with steps or segments
Stations with segmented data or step changes in the signal, requiring special processing.

In [ ]:
all_stats["n_steps"] = 0 # init
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if len(t_.segments) > 0:
        print(station, len(t_.segments), "segments")
        all_stats.loc[all_stats.ioc_code == station, "n_steps"] = len(t_.segments)

(all_*all_stats[all_stats.n_steps>0].hvplot.points(
    x= "lon", y='lat', 
    c= 'n_steps',
    hover_cols = ['index'],
    coastline = True, 
    tiles = True, 
    line_color='r', 
    cmap = "fire",
    s = 70)).opts(width = 1200, height = 500)

### 5.6 Stations with Tsunami
Stations where tsunami events have been detected and flagged for further analysis.

In [ ]:
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if len(t_.tsunami) > 0:
        all_stats.loc[all_stats.ioc_code == station, "tsunamis"] = len(t_.tsunami)

(all_*all_stats[all_stats.tsunamis>0].hvplot.points(
    x= "lon", y='lat', 
    c= 'tsunamis',
    hover_cols = ['index'],
    coastline = True, 
    tiles = True, 
    line_color='k',
    cmap = "kbc",
    s = 150)).opts(width = 1200, height = 500)

#### example with the Stromboli (2nd of July 2024)
there seemed to be a tsunami generated by Stromboli activity in the south Tyrrenian sea

In [ ]:
tsunami_start = pd.Timestamp(2024,7,1)
tsunami_end = pd.Timestamp(2024,7,5)

for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")    
    t_ = C.load_transformation_from_path(f"../transformations/{stations_sensor}.json")
    if len(t_.tsunami) > 0:
        for seg in t_.tsunami:
            if (seg[0]>tsunami_start) and seg[1]<tsunami_end:
                try:
                    df = read(station, sensor)
                    c_ = clean(station, sensor)
                    c_ = c_.loc["2024-01-01":"2024-12-31"].dropna()
                    lat = ioc_cleanup_2024[ioc_cleanup_2024.ioc_code == station].lat.values[0]
                    s_ = surge(c_, lat, rsmp=2)
                    s_ = s_.loc[tsunami_start:tsunami_end]
                    all_stats.loc[all_stats.ioc_code == station, "runup"] = s_.max()
                except Exception as e:
                    print(f"Error reading {station} {sensor} {e}")

In [ ]:
(all_*all_stats[all_stats.runup>0].hvplot.points(
    x= "lon", y='lat', 
    c= 'runup',
    hover_cols = ['index'],
    coastline = True, 
    tiles = True, 
    line_color='k',
    cmap = "kbc",
    s = 150)).opts(width = 1200, height = 500, title = "max runup from tsunami")

### 5.7 - DART buoys

DART (Deep-Ocean Assessment and Reporting of Tsunamis) buoys are excluded from tide/surge analysis as they serve a different purpose.

In [ ]:
for stations_sensor in sorted(stations_sensors): 
    station, sensor = stations_sensor.split("_")
    if "prt" in stations_sensor:
        all_stats.loc[all_stats.ioc_code == station, "dart"] = True

(all_ * all_stats[all_stats.dart == True].hvplot.points(
    x= "lon", y='lat', 
    color="violet",
    coastline = True, 
    tiles = True, 
    line_color='k',
    colorbar=False,
    label = 'final',
    s = 200)
).opts(width = 1200, height = 500)

### 5.8 - the final stations
A map of stations that have been successfully processed and are ready for analysis.

In [ ]:
final = all_stats[np.logical_and(all_stats.processed == True, all_stats.skip!=True)].hvplot.points(
    x= "lon", y='lat', 
    color="green",
    hover_cols = ['index'], 
    coastline = True, 
    tiles = True, 
    line_color='k',
    colorbar=False,
    label = 'final',
    s = 200)

(all_*final).opts(width = 1200, height = 500)

## 6.0 - Results

### 6.1 - Show the most modified stations

This section highlights stations with the highest percentage of discarded data

In [ ]:
all_stats

In [ ]:
pattern = r"transformations/([\w\d]+)\.json"
stations_sensors = [re.search(pattern, path).group(1) for path in glob.glob("../transformations/*json")]
for station_sensor in sorted(stations_sensors): 
    station, sensor = station_sensor.split("_")    
    item = all_stats[all_stats.ioc_code == station]
    if not item.empty:
        print(item.skip)
        t_ = C.load_transformation_from_path(f"../transformations/{station_sensor}.json")
        r_ = read(station, sensor)
        c_ = clean(station, sensor)
        all_stats.loc[all_stats.ioc_code == station, "perc_removed"] = (r_.count()-c_.count())/r_.count()

In [ ]:
all_stats.sort_values("perc_removed", ascending=False)

### 6.2 - save plots locally
Save cleaned and processed data visualizations as HTML files for offline review.

In [ ]:
for ii, item in all_stats[all_stats.processed==True].sort_values("perc_removed", ascending=False).iterrows():
    station, sensor = item.ioc_code, item.sensor
    print(station, sensor, (not item.skip) and (item.processed), item.perc_removed)
    if (not item.skip) and (item.processed):
        lat = ioc_cleanup_2024[ioc_cleanup_2024.ioc_code == station].lat.values[0]
        try:
            p_ = (
                read(station, sensor).dropna().hvplot(label='raw', c='r')*\
                clean(station, sensor).dropna().hvplot(label='clean', c='b')
            ).opts(width= 1500, height=800)
            hv.save(p_, f"html/{station}_{sensor}_{np.round(item.perc_removed, 4)}.html")
        except Exception as e:
            print(f"Error with {station}_{sensor}: {e}")

In [ ]:
perc_rm = all_stats.hvplot.points(
    x= "lon", y='lat', 
    hover_cols = ['index',"perc_removed" ],
    color = "perc_removed", 
    coastline = True,
    tiles = True,
    s = 200, line_color = 'grey', cmap="rainbow_r").opts(width = 1300, height = 600)
perc_rm

### 6.3 - save surges
Save detided surge data for further analysis or external use.

In [ ]:
for ii, item in all_stats[all_stats.processed==True].sort_values("perc_removed", ascending=False).iterrows():
    station, sensor = item.ioc_code, item.sensor
    print(station, sensor, item.perc_removed)
    if not item.skip:
        lat = ioc_cleanup_2024[ioc_cleanup_2024.ioc_code == station].lat.values[0]
        ts = surge(clean(station, sensor), lat, rsmp = 2)
        ts.to_frame(name = sensor).to_parquet(f"../surge/{station}_{sensor}.parquet")